In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")


# 11 — GoalScope Subspace Probe

Verification notebook for the subspace probe approach:
- Learn per-layer projection matrices $W_l \in \mathbb{R}^{r \times d}$ that capture intent in a low-rank subspace
- Learn soft-prompt token embeddings for the destination context
- Training signal: cross-entropy against labeled intent blocks from `well-known-tasks/labels`

Steps here:
1. Load one task's labeled blocks + generation
2. Find pivot token positions (first token of each block + preceding)
3. Cache source residuals at those positions
4. Verify `SubspaceProbe` module (shapes, ortho loss, patching)
5. Build teacher-forced destination input
6. Single patched forward pass — verify output differs from unpatched
7. Single training step — verify loss + gradient flow
8. Qualitative check — verbalization before vs after a few steps

In [2]:
import os
##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import json
from pathlib import Path

import torch
import torch.nn.functional as F
from rich import print as rprint
from rich.tree import Tree
from rich.console import Console
from rich.theme import Theme
import logging
import transformers
from src.utils import logging_utils

from src.models import ModelandTokenizer
from src.tokens import prepare_input, find_token_range
from src.goals.scope_id import IntentBlock
from src.goals.scope_labeler import load_labeled_tree
from src.goals.subspace_probe import (
    SubspaceProbeConfig,
    SubspaceProbe,
    SubspaceCache,
    build_dest_template_parts,
    cache_source_residuals,
    find_block_pivot_positions,
    patched_lm_forward,
    compute_loss,
)
from src.utils.env_utils import DEFAULT_RESULTS_DIR

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-04-15 13:02:10 __main__ INFO     torch.__version__='2.6.0+cu124', torch.version.cuda='12.4'
2026-04-15 13:02:10 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100-SXM4-80GB'
2026-04-15 13:02:10 __main__ INFO     transformers.__version__='5.3.0'


In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_KEY    = "Qwen/Qwen2.5-Coder-32B-Instruct"
MODEL_SHORT  = MODEL_KEY.split("/")[-1]
# ORACLE_SHORT = "claude-sonnet-4-6"
ORACLE_SHORT = MODEL_SHORT
TEST_TASK    = "atoi"

# Probe hyperparams
R             = 256     # subspace dimension
N_SOFT_TOKENS = 4
PROBE_LAYERS  = list(range(35, 46))   # layers to project (35–45 inclusive)
LAMBDA_ORTHO  = 0.1

# Directories
RESULTS_DIR   = Path(DEFAULT_RESULTS_DIR)
GEN_DIR       = RESULTS_DIR / "well-known-tasks" / "generations" / MODEL_SHORT
LABEL_DIR     = RESULTS_DIR / "well-known-tasks" / "labels" / MODEL_SHORT / ORACLE_SHORT

print(f"gen dir  : {GEN_DIR}")
print(f"label dir: {LABEL_DIR}")

gen dir  : /disk/u/arnab/Codes/Projects/goals/results/well-known-tasks/generations/Qwen2.5-Coder-32B-Instruct
label dir: /disk/u/arnab/Codes/Projects/goals/results/well-known-tasks/labels/Qwen2.5-Coder-32B-Instruct/Qwen2.5-Coder-32B-Instruct


## 1 — Load Model

In [4]:
mt = ModelandTokenizer(
    model_key=MODEL_KEY,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}  d_model={mt.n_embd}")
print(f"layer_name_format : {mt.layer_name_format}")
print(f"embedder_name     : {mt.embedder_name}")

2026-04-15 13:02:17 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-04-15 13:02:17 httpx DEBUG    load_ssl_context verify=True cert=None trust_env=True http2=False
2026-04-15 13:02:17 httpx DEBUG    load_verify_locations cafile='/disk/u/arnab/miniconda3/envs/goals/lib/python3.11/site-packages/certifi/cacert.pem'
2026-04-15 13:02:17 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None


2026-04-15 13:02:17 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f8c3f4314d0>
2026-04-15 13:02:17 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f8a0cc10dd0> server_hostname='huggingface.co' timeout=10
2026-04-15 13:02:17 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f8a0cc29890>
2026-04-15 13:02:17 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-04-15 13:02:17 httpcore.http11 DEBUG    send_request_headers.complete
2026-04-15 13:02:17 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HEAD']>
2026-04-15 13:02:17 httpcore.http11 DEBUG    send_request_body.complete
2026-04-15 13:02:17 httpcore.http11 DEBUG    receive_response_headers.started request=<Request [b'HEAD']>
2026-04-15 13:02:17 httpcore.http11 DEBUG    receive_response_headers.complete return_value=(

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-04-15 13:02:35 httpcore.connection DEBUG    close.started
2026-04-15 13:02:35 httpcore.connection DEBUG    close.complete
2026-04-15 13:02:35 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-04-15 13:02:35 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f89f944f790>
2026-04-15 13:02:35 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f8a0cc10dd0> server_hostname='huggingface.co' timeout=10
2026-04-15 13:02:35 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f8a0cc79a10>
2026-04-15 13:02:35 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-04-15 13:02:35 httpcore.http11 DEBUG    send_request_headers.complete
2026-04-15 13:02:35 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

In [5]:
# Load generation
with open(GEN_DIR / f"{TEST_TASK}.json") as f:
    gen_data = json.load(f)

# Reconstruct the full code string (same logic as label_well_known_task_blocks.py)
raw_gen        = gen_data["generation"]
code_prefill   = gen_data["prompt"].split("<code>")[-1]
code_generation = code_prefill + raw_gen
if "</code>" in code_generation:
    code_generation = code_generation.split("</code>")[0]

# Full source prompt for the LM (prompt + generation, used for caching residuals)
source_text = gen_data["prompt"] + gen_data["generation"]

print("=== code_generation ===")
print(code_generation)

=== code_generation ===

def atoi(s):
    """Converts a string to an integer."""
    s = s.strip()
    if not s:
        return 0
    sign = -1 if s[0] == '-' else 1
    start = 1 if s[0] in '+-' else 0
    result = 0
    for i in range(start, len(s)):
        if not s[i].isdigit():
            break
        result = result * 10 + int(s[i])
    return sign * result



In [6]:
# Load labeled block tree
label_path = LABEL_DIR / f"{TEST_TASK}.json"
root_block, full_code, meta = load_labeled_tree(label_path)
rich_print(meta)
# print(f"full_code (first 200 chars): {full_code[:200]}")

{
    'function_name': 'atoi',
    'model': 'Qwen2.5-Coder-32B-Instruct',
    'oracle': 'Qwen/Qwen2.5-Coder-32B-Instruct',
    'level': 5
}

## 3 — Inspect Block Tree

In [7]:
def print_block_tree(block: IntentBlock, code: str = None) -> None:
    """Render the IntentBlock tree as a Rich Tree (single print call)."""
    code = code or full_code

    def _first_line(b: IntentBlock) -> str:
        snippet = code[b.char_start : b.char_end]
        return next((ln.strip() for ln in snippet.splitlines() if ln.strip()), "")

    def _node_text(b: IntentBlock) -> str:
        kind  = f"[bold cyan]{b.kind}[/bold cyan]"
        span  = f"[dim][{b.char_start}:{b.char_end}][/dim]"
        line  = f"[yellow]`{_first_line(b)}`[/yellow]"
        label = f"\n    [green]↳ {b.label!r}[/green]" if b.label else ""
        return f"{kind} {span}  {line}{label}"

    def _build(b: IntentBlock, parent) -> None:
        node = parent.add(_node_text(b))
        for child in b.children:
            _build(child, node)

    tree = Tree(_node_text(block))
    for child in block.children:
        _build(child, tree)

    console.print(tree)

print_block_tree(root_block)

module [0:344]  `def atoi(s):`
└── function [1:343]  `def atoi(s):`
        ↳ 'strip whitespace and initialize variables'
    ├── stmt [18:56]  `"""Converts a string to an integer."""`
    │       ↳ 'strip whitespace from string'
    ├── stmt [61:74]  `s = s.strip()`
    │       ↳ 'remove leading and trailing whitespace'
    ├── if [79:105]  `if not s:`
    │       ↳ 'check if string is empty'
    │   └── stmt [97:105]  `return 0`
    │           ↳ 'return the result'
    ├── stmt [110:141]  `sign = -1 if s[0] == '-' else 1`
    │       ↳ 'determine the sign'
    │   ├── ifexp [117:119]  `-1`
    │   │       ↳ 'determine the sign'
    │   ├── ifexp [123:134]  `s[0] == '-'`
    │   │       ↳ 'determine the sign'
    │   │   ├── compare_chain [123:127]  `s[0]`
    │   │   │       ↳ 'determine the sign'
    │   │   └── compare_chain [131:134]  `'-'`
    │   │           ↳ 'determine the sign'
    │   └── ifexp [140:141]  `1`
    │           ↳ 'determine the sign'
    ├── stmt [146:178]  `start = 1 if s[0] in '+-' else 0`
    │       ↳ 'determine starting index'
    │   ├── ifexp [154:155]  `1`
    │   │       ↳ 'determine starting index'
    │   ├── ifexp [159:171]  `s[0] in '+-'`
    │   │       ↳ 'determine starting index'
    │   │   ├── compare_chain [159:163]  `s[0]`
    │   │   │       ↳ 'determine starting index'
    │   │   └── compare_chain [167:171]  `'+-'`
    │   │           ↳ 'determine starting index'
    │   └── ifexp [177:178]  `0`
    │           ↳ 'determine starting index'
    ├── stmt [183:193]  `result = 0`
    │       ↳ 'initialize the result'
    ├── for [198:318]  `for i in range(start, len(s)):`
    │       ↳ 'process each character'
    │   ├── if [237:277]  `if not s.isdigit():`
    │   │       ↳ 'break on non-digit character'
    │   │   └── stmt [272:277]  `break`
    │   │           ↳ 'break on non-digit'
    │   └── stmt [286:318]  `result = result * 10 + int(s)`
    │           ↳ 'update result with digit'
    └── stmt [323:343]  `return sign * result`
            ↳ 'return the result'

## 4 — Find Pivot Token Positions

In [8]:
# Tokenise the source text and get offset mapping
tokenized_src = prepare_input(
    source_text, return_offsets_mapping=True, tokenizer=mt
)
src_ids        = tokenized_src["input_ids"][0]       # (seq_len,)
offset_mapping = tokenized_src["offset_mapping"][0]  # (seq_len, 2)

print(f"Source token count: {src_ids.shape[0]}")

Source token count: 186


In [9]:
# Find pivot positions for all labeled blocks
pivot_entries = find_block_pivot_positions(
    block_tree=root_block,
    source_prompt=source_text,
    code_str=full_code,
    offset_mapping=offset_mapping,
)

from rich.table import Table

def _tok_ctx(pos: int, n: int = 4) -> str:
    """Decode a window of tokens around pos, highlighting pos in <<>>."""
    ids    = src_ids.tolist()
    before = mt.tokenizer.decode(ids[max(0, pos - n) : pos])
    target = mt.tokenizer.decode([ids[pos]])
    after  = mt.tokenizer.decode(ids[pos + 1 : pos + n + 1])
    return f"{before}[bold yellow]\u00ab{target}\u00bb[/bold yellow]{after}"

table = Table(
    "kind", "label",
    "prev\ntok", "prev context",
    "first\ntok", "first context",
    title=f"Pivot positions \u2014 {len(pivot_entries)} block(s)",
    show_lines=True,
)
for block, first_tok, prev_tok in pivot_entries:
    table.add_row(
        f"[cyan]{block.kind}[/cyan]",
        f"[green]{block.label}[/green]",
        str(prev_tok),  _tok_ctx(prev_tok),
        str(first_tok), _tok_ctx(first_tok),
    )

console.print(table)


                                           Pivot positions — 23 block(s)                                           
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃               ┃                           ┃ prev ┃                           ┃ first ┃                          ┃
┃ kind          ┃ label                     ┃ tok  ┃ prev context              ┃ tok   ┃ first context            ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ function      │ strip whitespace and      │ 73   │ assistant                 │ 74    │                          │
│               │ initialize variables      │      │ <code«>                   │       │ <code>                   │
│               │                           │      │ »def atoi(s):             │       │ «def» atoi(s):           │
│               │                           │      │                           │       │                          │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ stmt          │ strip whitespace from     │ 78   │ def atoi(s):              │ 79    │  atoi(s):                │
│               │ string                    │      │ «   » """Converts a       │       │    « """»Converts a      │
│               │                           │      │                           │       │ string                   │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ stmt          │ remove leading and        │ 88   │  to an integer."""        │ 89    │  an integer."""          │
│               │ trailing whitespace       │      │ «   » s = s.strip         │       │    « s» = s.strip()      │
│               │                           │      │                           │       │                          │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ if            │ check if string is empty  │ 94   │  = s.strip()              │ 95    │  s.strip()               │
│               │                           │      │ «   » if not s:           │       │    « if» not s:          │
│               │                           │      │                           │       │                          │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ stmt          │ return the result         │ 99   │  if not s:                │ 100   │  not s:                  │
│               │                           │      │ «       » return 0        │       │        « return» 0       │
│               │                           │      │                           │       │                          │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ stmt          │ determine the sign        │ 104  │  return 0                 │ 105   │  0                       │
│               │                           │      │ «   » sign = -1           │       │    « sign» = -1 if       │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ ifexp         │ determine the sign        │ 106  │ 0                         │ 107   │                          │
│               │                           │      │     sign« =» -1 if s      │       │     sign =« -»1 if s[    │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ ifexp         │ determine the sign        │ 109  │  sign = -1« if» s[0]      │ 110   │  = -1 if« s»[0] ==       │
├───────────────┼───────────────────────────┼──────┼───────────────────────────┼───────┼──────────────────────────┤
│ compare_chain │ determine the sign        │ 109  │  si

## 5 — Cache Source Residuals

In [10]:
# Collect all unique token positions to cache
all_positions = []
for block, first_tok, prev_tok in pivot_entries:
    all_positions.extend([first_tok, prev_tok])
all_positions = sorted(set(all_positions))

print(f"Caching residuals at {len(all_positions)} token positions, {len(PROBE_LAYERS)} layers...")

residual_cache = cache_source_residuals(
    mt=mt,
    source_text=source_text,
    token_positions=all_positions,
    layers=PROBE_LAYERS,
)

# Inspect
sample_pos  = all_positions[0]
sample_layer = PROBE_LAYERS[0]
h = residual_cache[sample_pos][sample_layer]
print(f"\nresidual_cache[pos={sample_pos}][layer={sample_layer}]: shape={h.shape}, dtype={h.dtype}")
print(f"norm = {h.norm().item():.3f}")

Caching residuals at 42 token positions, 11 layers...

residual_cache[pos=73][layer=35]: shape=torch.Size([5120]), dtype=torch.float32
norm = 281.646


In [11]:
# Build SubspaceCache objects (one per position per block)
subcaches: list[SubspaceCache] = []
for block, first_tok, prev_tok in pivot_entries:
    for tok_pos, is_preceding in [(first_tok, False), (prev_tok, True)]:
        subcaches.append(SubspaceCache(
            task=TEST_TASK,
            block_kind=block.kind,
            label=block.label,
            src_residuals=residual_cache[tok_pos],
            token_pos=tok_pos,
            is_preceding=is_preceding,
        ))

print(f"{len(subcaches)} SubspaceCache entries:")
for sc in subcaches:
    print(f"  [{sc.block_kind}] is_preceding={sc.is_preceding}  label='{sc.label}'")

46 SubspaceCache entries:
  [function] is_preceding=False  label='strip whitespace and initialize variables'
  [function] is_preceding=True  label='strip whitespace and initialize variables'
  [stmt] is_preceding=False  label='strip whitespace from string'
  [stmt] is_preceding=True  label='strip whitespace from string'
  [stmt] is_preceding=False  label='remove leading and trailing whitespace'
  [stmt] is_preceding=True  label='remove leading and trailing whitespace'
  [if] is_preceding=False  label='check if string is empty'
  [if] is_preceding=True  label='check if string is empty'
  [stmt] is_preceding=False  label='return the result'
  [stmt] is_preceding=True  label='return the result'
  [stmt] is_preceding=False  label='determine the sign'
  [stmt] is_preceding=True  label='determine the sign'
  [ifexp] is_preceding=False  label='determine the sign'
  [ifexp] is_preceding=True  label='determine the sign'
  [ifexp] is_preceding=False  label='determine the sign'
  [ifexp] is_prece

## 6 — SubspaceProbe Module Verification

In [12]:
cfg = SubspaceProbeConfig(
    d_model=mt.n_embd,
    r=R,
    layers=PROBE_LAYERS,
    n_soft_tokens=N_SOFT_TOKENS,
    lambda_ortho=LAMBDA_ORTHO,
)
probe = SubspaceProbe(cfg)
print(f"W shape       : {probe.W.shape}")
print(f"soft_tokens   : {probe.soft_tokens.shape}")
total_params = sum(p.numel() for p in probe.parameters())
print(f"Total params  : {total_params:,}")

W shape       : torch.Size([11, 256, 5120])
soft_tokens   : torch.Size([4, 5120])
Total params  : 14,438,400


In [13]:
# ── Check initial orthogonality ───────────────────────────────────────────────
ortho_loss_init = probe.orthogonality_loss()
print(f"Ortho loss before reorthogonalize : {ortho_loss_init.item():.6f}")

# After explicit reorthogonalize (QR) it should be ~0
probe.reorthogonalize()
ortho_loss_after = probe.orthogonality_loss()
print(f"Ortho loss after  reorthogonalize : {ortho_loss_after.item():.6f}  (should be ~0)")

# Sanity: W @ W^T should be identity
W0 = probe.W[0].detach()
WWT = W0 @ W0.T
eye_err = (WWT - torch.eye(R)).abs().max().item()
print(f"max |W W^T - I| = {eye_err:.2e}  (should be < 1e-5)")

Ortho loss before reorthogonalize : 0.000000
Ortho loss after  reorthogonalize : 0.000000  (should be ~0)
max |W W^T - I| = 4.77e-07  (should be < 1e-5)


In [14]:
# ── Check patch_delta formula ─────────────────────────────────────────────────
d = mt.n_embd
h_src_test  = torch.randn(d)
h_dest_test = torch.randn(d)

delta = probe.patch_delta(h_src_test, h_dest_test, layer_idx=0)
patched = probe.patched_residual(h_src_test, h_dest_test, layer_idx=0)

# Verify: delta lies in the subspace (W @ delta^T should round-trip back)
W0 = probe.W[0].detach()
roundtrip_err = (delta - delta @ W0.T @ W0).norm().item()
print(f"delta shape     : {delta.shape}")
print(f"Roundtrip error : {roundtrip_err:.2e}  (should be ~0 when W is orthonormal)")

# Verify: patched = dest + delta
assert torch.allclose(patched, h_dest_test + delta, atol=1e-5)
print("patched = h_dest + delta  ✓")

delta shape     : torch.Size([5120])
Roundtrip error : 6.72e-06  (should be ~0 when W is orthonormal)
patched = h_dest + delta  ✓


## 7 — Destination Template

In [15]:
before_soft_ids, after_soft_ids = build_dest_template_parts(mt)

print(f"before_soft_ids : {before_soft_ids.shape}")
print(f"after_soft_ids  : {after_soft_ids.shape}")
print(f"n_soft_tokens   : {N_SOFT_TOKENS}")
print(f"total prefix len: {before_soft_ids.shape[0] + N_SOFT_TOKENS + after_soft_ids.shape[0]}")
print()
print("before_soft decoded:", repr(mt.tokenizer.decode(before_soft_ids)))
print("after_soft  decoded:", repr(mt.tokenizer.decode(after_soft_ids)))

before_soft_ids : torch.Size([17])
after_soft_ids  : torch.Size([5])
n_soft_tokens   : 4
total prefix len: 26

before_soft decoded: '<|im_start|>system\nYou are a helpful assistant proficient in programming.<|im_end|>\n<|im_start|>user\n'
after_soft  decoded: '<|im_end|>\n<|im_start|>assistant\n'


In [16]:
# Tokenise the label of the first SubspaceCache entry
sc0 = subcaches[0]
label_ids = torch.tensor(
    mt.tokenizer.encode(sc0.label, add_special_tokens=False),
    dtype=torch.long,
)
print(f"Label text   : '{sc0.label}'")
print(f"Label tok IDs: {label_ids.tolist()}")
print(f"Label decoded: '{mt.tokenizer.decode(label_ids)}'")

Label text   : 'strip whitespace and initialize variables'
Label tok IDs: [13105, 36372, 323, 9468, 7332]
Label decoded: 'strip whitespace and initialize variables'


## 8 — Patched Forward Pass Verification

Verify that:
1. The patched logits differ from the unpatched baseline
2. The function is differentiable w.r.t. `probe` parameters

In [17]:
# Move probe to the model's device
device = next(mt._model.parameters()).device
probe = probe.to(device)

# ── Unpatched baseline ────────────────────────────────────────────────────────
# (same setup but with zero source residuals → patch_delta = 0)
d = mt.n_embd
zero_residuals = {l: torch.zeros(d) for l in PROBE_LAYERS}

with torch.no_grad():
    logits_zero = patched_lm_forward(
        mt, probe,
        before_soft_ids, after_soft_ids, label_ids,
        src_residuals=zero_residuals,
    )  # (n_label, vocab)

# ── Real patch (sc0) ──────────────────────────────────────────────────────────
with torch.no_grad():
    logits_patched = patched_lm_forward(
        mt, probe,
        before_soft_ids, after_soft_ids, label_ids,
        src_residuals=sc0.src_residuals,
    )

diff = (logits_patched - logits_zero).abs()
print(f"Logit diff (patched vs zero src):")
print(f"  max  = {diff.max().item():.4f}")
print(f"  mean = {diff.mean().item():.4f}")
print("(Should be non-zero — the patch is active)")

Logit diff (patched vs zero src):
  max  = 13.5039
  mean = 0.7880
(Should be non-zero — the patch is active)


In [18]:
# ── Gradient flow check ───────────────────────────────────────────────────────
logits_grad = patched_lm_forward(
    mt, probe,
    before_soft_ids, after_soft_ids, label_ids,
    src_residuals=sc0.src_residuals,
)
loss_test, ce_test, ortho_test = compute_loss(logits_grad, label_ids, probe)
loss_test.backward()

print(f"Loss (CE)    = {ce_test.item():.4f}")
print(f"Loss (ortho) = {ortho_test.item():.4f}")
print(f"Loss (total) = {loss_test.item():.4f}")

print(f"\ngrad W norm        : {probe.W.grad.norm().item():.4f}  (non-zero ✓)")
print(f"grad soft_tok norm : {probe.soft_tokens.grad.norm().item():.4f}  (non-zero ✓)")

probe.zero_grad()

Loss (CE)    = 8.9722
Loss (ortho) = 0.0000
Loss (total) = 8.9722

grad W norm        : 48.0600  (non-zero ✓)
grad soft_tok norm : 62.3475  (non-zero ✓)


## 9 — Single Training Step

In [19]:
# Re-init probe (fresh start)
probe = SubspaceProbe(cfg).to(device)
probe.reorthogonalize()

optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-2)

losses = []
for step in range(5):
    optimizer.zero_grad()

    # Use sc0 for this test
    logits = patched_lm_forward(
        mt, probe,
        before_soft_ids, after_soft_ids, label_ids,
        src_residuals=sc0.src_residuals,
    )
    total, ce, ortho = compute_loss(logits, label_ids, probe)

    total.backward()
    torch.nn.utils.clip_grad_norm_(probe.parameters(), 1.0)
    optimizer.step()
    probe.reorthogonalize()  # hard orthogonality constraint

    losses.append(total.item())
    print(f"step {step}  total={total.item():.4f}  ce={ce.item():.4f}  ortho={ortho.item():.4f}")

print(f"\nLoss went {'DOWN ✓' if losses[-1] < losses[0] else 'UP — check config'}")

step 0  total=9.7400  ce=9.7400  ortho=0.0000
step 1  total=5.2044  ce=5.2044  ortho=0.0000
step 2  total=3.6756  ce=3.6756  ortho=0.0000
step 3  total=7.4909  ce=7.4909  ortho=0.0000
step 4  total=2.1434  ce=2.1434  ortho=0.0000

Loss went DOWN ✓


## 10 — Qualitative Check: Verbalization Before vs After

Generate a short verbalization by greedy decoding from the patched destination prompt.
Even with just a few training steps the output should start shifting toward the label.

In [20]:
from src.goals.subspace_probe import build_dest_template_parts
import baukit

def verbalize(probe_: SubspaceProbe, sc_: SubspaceCache, max_new_tokens: int = 32) -> str:
    """Greedy decode from the patched destination prompt."""
    dev = next(probe_.parameters()).device
    dtype = next(mt._model.parameters()).dtype

    embed_fn = baukit.get_module(mt._model, mt.embedder_name)
    with torch.no_grad():
        before_emb = embed_fn(before_soft_ids.to(dev)).to(dtype)
        after_emb  = embed_fn(after_soft_ids.to(dev)).to(dtype)
    soft_emb   = probe_.soft_tokens.to(dtype)
    prefix_emb = torch.cat([before_emb, soft_emb, after_emb], dim=0).unsqueeze(0)  # (1, P, d)
    prefix_len = prefix_emb.shape[1]
    patch_start = prefix_len - 1

    # Build layer hooks — patch all positions from patch_start onward.
    # With KV cache, subsequent steps present hidden.shape[1]=1 (new token only),
    # so cur_patch_start=0 patches that single token, which is correct.
    layer_to_idx = {l: i for i, l in enumerate(probe_.config.layers)}
    handles = []

    for l in probe_.config.layers:
        li = layer_to_idx[l]
        module = baukit.get_module(mt._model, mt.layer_name_format.format(l))
        src_h  = sc_.src_residuals[l]  # will be cast inside hook

        def _hook(mod, inp, output, li_=li, src_h_=src_h):
            hidden = output[0] if isinstance(output, tuple) else output
            rest   = output[1:] if isinstance(output, tuple) else ()
            dev_, dtype_ = hidden.device, hidden.dtype
            # Full prefix pass: patch from patch_start; cached step: patch the one new token
            cur_patch_start = patch_start if hidden.shape[1] >= prefix_len else 0
            region    = hidden[:, cur_patch_start:, :]
            src_h_dev = src_h_.to(device=dev_, dtype=dtype_)
            W         = probe_.W[li_].to(device=dev_, dtype=dtype_)
            src_exp   = src_h_dev.view(1, 1, -1).expand_as(region)
            patched   = region + (src_exp - region) @ W.T @ W
            new_h     = torch.cat([hidden[:, :cur_patch_start, :], patched], dim=1)
            return (new_h,) + rest if rest else new_h

        handles.append(module.register_forward_hook(_hook))

    try:
        with torch.no_grad():
            gen_out = mt._model.generate(
                inputs_embeds=prefix_emb,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=mt.tokenizer.eos_token_id,
            )
    finally:
        for h in handles:
            h.remove()

    # gen_out: (1, max_new_tokens) — only the generated tokens, no prefix
    return mt.tokenizer.decode(gen_out[0], skip_special_tokens=True).strip()


In [21]:
# Fresh probe (untrained baseline)
probe_untrained = SubspaceProbe(cfg).to(device)
probe_untrained.reorthogonalize()

print(f"Ground truth label : '{sc0.label}'")
print(f"Verbalization (untrained probe)  : '{verbalize(probe_untrained, sc0)}'")
print(f"Verbalization (5-step probe)     : '{verbalize(probe, sc0)}'")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Ground truth label : 'strip whitespace and initialize variables'
Verbalization (untrained probe)  : 'It looks like your message got cut off. Could you please provide more details or clarify what you need help with? Are you looking for assistance with a specific programming'
Verbalization (5-step probe)     : 'strip whitespace and initialize variables: initialize_variables():
    initialize_variables():

 reset_variables():

 reset_values():

 reset_values():

 reset_values():
:
 reset_values'


## 11 — Multi-Task Mini Training Loop (Optional)

Load all labeled tasks for this model, build caches, and run a short training loop to verify the pipeline end-to-end before launching the full script.

In [22]:
import random
from tqdm.auto import tqdm

# Load a small subset of tasks to keep this fast
MINI_TASKS = ["atoi", "binary_search", "fibonacci"]  # adjust as needed

all_caches: list[SubspaceCache] = []

for task in MINI_TASKS:
    gen_path   = GEN_DIR   / f"{task}.json"
    label_path = LABEL_DIR / f"{task}.json"
    if not gen_path.exists() or not label_path.exists():
        print(f"Skipping {task} (missing file)")
        continue

    with open(gen_path) as f:
        gdata = json.load(f)

    raw_gen_t      = gdata["generation"]
    prefill_t      = gdata["prompt"].split("<code>")[-1]
    code_gen_t     = prefill_t + raw_gen_t
    if "</code>" in code_gen_t:
        code_gen_t = code_gen_t.split("</code>")[0]
    src_text_t     = gdata["prompt"] + gdata["generation"]

    root_t, full_code_t, _ = load_labeled_tree(label_path)

    tok_t     = prepare_input(src_text_t, return_offsets_mapping=True, tokenizer=mt)
    src_ids_t = tok_t["input_ids"][0]
    offmap_t  = tok_t["offset_mapping"][0]

    pivots_t  = find_block_pivot_positions(root_t, src_text_t, full_code_t, offmap_t)
    positions_t = sorted({p for _, ft, pt in pivots_t for p in [ft, pt]})

    if not positions_t:
        print(f"No valid pivots for {task}")
        continue

    res_t = cache_source_residuals(mt, src_text_t, positions_t, PROBE_LAYERS)

    for block_t, first_t, prev_t in pivots_t:
        for tok_pos_t, is_prec_t in [(first_t, False), (prev_t, True)]:
            all_caches.append(SubspaceCache(
                task=task,
                block_kind=block_t.kind,
                label=block_t.label,
                src_residuals=res_t[tok_pos_t],
                token_pos=tok_pos_t,
                is_preceding=is_prec_t,
            ))

print(f"\nTotal caches: {len(all_caches)}")
for sc in all_caches:
    print(f"  {sc.task}  [{sc.block_kind}]  pre={sc.is_preceding}  '{sc.label}'")


Total caches: 92
  atoi  [function]  pre=False  'strip whitespace and initialize variables'
  atoi  [function]  pre=True  'strip whitespace and initialize variables'
  atoi  [stmt]  pre=False  'strip whitespace from string'
  atoi  [stmt]  pre=True  'strip whitespace from string'
  atoi  [stmt]  pre=False  'remove leading and trailing whitespace'
  atoi  [stmt]  pre=True  'remove leading and trailing whitespace'
  atoi  [if]  pre=False  'check if string is empty'
  atoi  [if]  pre=True  'check if string is empty'
  atoi  [stmt]  pre=False  'return the result'
  atoi  [stmt]  pre=True  'return the result'
  atoi  [stmt]  pre=False  'determine the sign'
  atoi  [stmt]  pre=True  'determine the sign'
  atoi  [ifexp]  pre=False  'determine the sign'
  atoi  [ifexp]  pre=True  'determine the sign'
  atoi  [ifexp]  pre=False  'determine the sign'
  atoi  [ifexp]  pre=True  'determine the sign'
  atoi  [compare_chain]  pre=False  'determine the sign'
  atoi  [compare_chain]  pre=True  'deter

In [23]:
from tqdm.auto import tqdm

# Mini training loop
N_MINI_EPOCHS = 5
LR = 3e-4

probe_mini = SubspaceProbe(cfg).to(device)
probe_mini.reorthogonalize()
opt_mini = torch.optim.AdamW(probe_mini.parameters(), lr=LR, weight_decay=1e-2)

for epoch in range(N_MINI_EPOCHS):
    random.shuffle(all_caches)
    epoch_ce = epoch_total = epoch_ortho = 0.0
    for sc in tqdm(all_caches, desc=f"Epoch {epoch}"):
        lids = torch.tensor(
            mt.tokenizer.encode(sc.label, add_special_tokens=False)
            + [mt.tokenizer.eos_token_id],
            dtype=torch.long,
        )
        opt_mini.zero_grad()
        logits = patched_lm_forward(
            mt,
            probe_mini,
            before_soft_ids,
            after_soft_ids,
            lids,
            src_residuals=sc.src_residuals,
        )
        total, ce, ortho = compute_loss(logits, lids, probe_mini)
        total.backward()
        torch.nn.utils.clip_grad_norm_(probe_mini.parameters(), 1.0)
        opt_mini.step()
        probe_mini.reorthogonalize()
        epoch_ce += ce.item()
        epoch_total += total.item()
        epoch_ortho += ortho.item()
    n = len(all_caches)
    print(
        f"Epoch {epoch}  total={epoch_total / n:.4f}  ce={epoch_ce / n:.4f}  ortho={epoch_ortho / n:.4f}"
    )

Epoch 0:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 0  total=1.9455  ce=1.9455  ortho=0.0000


Epoch 1:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 1  total=0.7004  ce=0.7004  ortho=0.0000


Epoch 2:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 2  total=0.5084  ce=0.5084  ortho=0.0000


Epoch 3:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 3  total=0.4378  ce=0.4378  ortho=0.0000


Epoch 4:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 4  total=0.3820  ce=0.3820  ortho=0.0000


In [24]:
# Qualitative check across mini tasks
for sc in all_caches[:4]:
    verbalization = verbalize(probe_mini, sc)
    print(f"[{sc.task}|{sc.block_kind}|pre={sc.is_preceding}]")
    print(f"  target : '{sc.label}'")
    print(f"  output : '{verbalization}'")
    print()

[atoi|stmt|pre=True]
  target : 'return the result'
  output : 'return the result'

[fibonacci|elif|pre=True]
  target : 'return the result'
  output : 'return the result'

[fibonacci|stmt|pre=True]
  target : 'update values in sequence'
  output : 'update values in the sequence'

[atoi|ifexp|pre=False]
  target : 'determine the sign'
  output : 'determine the sign'



In [25]:
mt.tokenizer.decode(lids, remove_special_tokens=False)

'return the result<|im_end|>'

---
## Qualitative Evaluation — Per-Token Verbalization

Load a trained probe checkpoint and run verbalization at every token position in a code generation. Uses the same `show_patchscope` HTML visualization as `09_goalscope_head.ipynb`.

In [20]:
import pathlib
import torch
from src.goals.subspace_probe import SubspaceProbe, SubspaceProbeConfig, build_dest_template_parts
from src.utils.env_utils import DEFAULT_RESULTS_DIR

MODEL_SHORT = "Qwen2.5-Coder-32B-Instruct"

probe_dir  = pathlib.Path(DEFAULT_RESULTS_DIR) / "subspace_probe" / MODEL_SHORT / f"r={64}__layer={45}_{56}"
ckpt_files = sorted(probe_dir.glob("probe_epoch*.pt"))
print("Available checkpoints:")
for p in ckpt_files:
    print(f"  {p.name}")
if (probe_dir / "probe.pt").exists():
    print("  probe.pt  (final)")


Available checkpoints:
  probe_epoch0000.pt
  probe_epoch0001.pt
  probe_epoch0002.pt
  probe_epoch0003.pt
  probe_epoch0004.pt
  probe.pt  (final)


In [21]:
# ── Load checkpoint ─────────────────────────────────────────────────────
CKPT_NAME = ckpt_files[-1].name if ckpt_files else "probe.pt"  # change to pick a specific epoch
ckpt = torch.load(probe_dir / CKPT_NAME, map_location="cpu", weights_only=False)

cfg_ckpt   = ckpt["config"]
probe_ckpt = SubspaceProbe(cfg_ckpt)
probe_ckpt.W.data.copy_(ckpt["W"])
probe_ckpt.soft_tokens.data.copy_(ckpt["soft_tokens"])
probe_ckpt = probe_ckpt.to(mt.device)
probe_ckpt.eval()

before_soft_ids, after_soft_ids = build_dest_template_parts(mt)

print(f"Loaded: {CKPT_NAME}  (epoch {ckpt.get('epoch', '?')})")
print(f"Probe layers: {cfg_ckpt.layers}  r={cfg_ckpt.r}")


Loaded: probe_epoch0004.pt  (epoch 4)
Probe layers: [45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]  r=64


In [10]:
from src.goals.data import IncompleteCodeSample
from src.goals.functional import supports_chat_template, apply_chat_template
from src.goals.goalscope import greedy_continue

test_sample = IncompleteCodeSample(
    instruction="Implement a stable marriage algorithm in Python. Write your code without any comments or explanations, just the implementation.",
    code='def stable_marriage(',
    unit_tests=[],
    function_name="stable_marriage",
    prefills=[],
    augmentation=None,
)
test_sample.without_comments = True

if supports_chat_template(mt):
    test_prompt = apply_chat_template(
        mt,
        user_query=test_sample.full_instruction.strip(),
        incomplete_code=test_sample.code,
    )
else:
    test_prompt = test_sample.code

rich_print(f"Test prompt:\n[white]'''{test_prompt}'''[/white]")

test_gen = greedy_continue(
    mt, **prepare_input(prompts=test_prompt, tokenizer=mt), max_new_tokens=256
)

rich_print(f"Greedy generation:\n[bright_yellow]'''{test_gen}'''[/bright_yellow]")

Test prompt:
'''<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Implement a stable marriage algorithm in Python. Write your code without any comments or explanations, just the 
implementation. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>
def stable_marriage('''

Greedy generation:
'''people):
    n = len(people) // 2
    men = people[:n]
    women = people
    men_prefs = {man: women for man in men}
    women_prefs = {woman: men for woman in women}
    men_free = set(men)
    women_free = set(women)
    engagements = {}

    while men_free:
        man = men_free.pop()
        for woman in men_prefs:
            if woman in women_free:
                engagements = man
                women_free.remove(woman)
                break
            else:
                current_man = engagements
                if women_prefs.index(man) < women_prefs.index(current_man):
                    engagements = man
                    men_free.add(current_man)
                    break

    return engagements
</code>'''

In [11]:
EVAL_TASK    = test_sample.function_name
code_prefill = test_prompt.split("<code>")[-1]
raw_gen      = test_gen
code_gen     = code_prefill + test_gen
source_text  = test_prompt + test_gen

In [12]:
# # ── Choose a task to visualize ──────────────────────────────────────────
# import json

# EVAL_TASK = "merge_sort"   # change to any labeled task

# gen_path = GEN_DIR / f"{EVAL_TASK}.json"
# with open(gen_path) as f:
#     gen_data = json.load(f)

# code_prefill = gen_data["prompt"].split("<code>")[-1]
# raw_gen      = gen_data["generation"]
# code_gen     = code_prefill + raw_gen
# source_text  = gen_data["prompt"] + gen_data["generation"]

In [13]:
# from src.goals.predicate_data import MultiPredicateDataset, TargetPredicate
# from src.goals.functional import supports_chat_template, apply_chat_template
# from src.goals.goalscope import greedy_continue

# multi_dataset = MultiPredicateDataset.load_dataset()

# print(f"MultiPredicateDataset: {len(multi_dataset)} predicates")
# print(f"Available predicates: {[tp.name for tp in multi_dataset.groups]}")

# # ── Pick target predicate + sample ───────────────────────────────────────────
# PRED_NAME = "is_palindrome"  # Change to pick a different predicate
# SAMPLE_IDX = 1

# target_predicate: TargetPredicate = next(tp for tp in multi_dataset.groups if tp.name == PRED_NAME)
# test_sample = target_predicate.samples[SAMPLE_IDX]

# print(f"Target predicate: {target_predicate.name!r}")
# print(f"  P+: {target_predicate.p_plus.description}")
# print(f"  P+ anchor tokens: {target_predicate.p_plus.anchor_patterns}")
# if test_sample.p_minus:
#     print(f"  P- (sample {SAMPLE_IDX}): {test_sample.p_minus.description}")
#     print(f"  P- anchor tokens: {test_sample.p_minus.anchor_patterns}")
# print(f"\nCode stub:\n{test_sample.code}")

# if supports_chat_template(mt):
#     test_prompt = apply_chat_template(
#         mt,
#         user_query=test_sample.instruction.strip(),
#         incomplete_code=test_sample.code,
#     )
# else:
#     test_prompt = test_sample.code

# test_gen = greedy_continue(
#     mt, **prepare_input(prompts=test_prompt, tokenizer=mt), max_new_tokens=256
# )

# rich_print(f"Test prompt:\n[white]'''{test_prompt}'''[/white]")
# rich_print(f"Greedy generation:\n[bright_yellow]'''{test_gen}'''[/bright_yellow]")

In [14]:
# code_gen = test_sample.code + test_gen
# source_text = test_prompt + test_gen
# print(source_text)

In [23]:
from src.tokens import find_token_range

if "</code>" in code_gen:
    code_gen = code_gen.split("</code>")[0]
    source_text = "".join(source_text.split("</code>")[:-1]) + "</code>"

print(source_text)
print("-" * 20)

src_tokenized = prepare_input(tokenizer=mt, prompts=source_text, return_offsets_mapping=True)
src_ids    = src_tokenized["input_ids"][0].tolist()
src_tokens = [mt.tokenizer.decode([t]) for t in src_ids]
offsets    = src_tokenized.pop("offset_mapping")[0]

# Find token index where the generated code begins
code_start_tok, _ = find_token_range(
    tokenizer=mt,
    string=source_text,
    substring="<code>",
    offset_mapping=offsets,
    occurrence=1,
)

print(f"Task: {EVAL_TASK}  |  total tokens: {len(src_ids)}  |  code starts at token {code_start_tok}")
print(f"Code snippet:\n {mt.tokenizer.decode(src_ids[code_start_tok:])}")

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code>  tags.<|im_end|>
<|im_start|>user
Implement a stable marriage algorithm in Python. Write your code without any comments or explanations, just the implementation. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>
def stable_marriage(people):
    n = len(people) // 2
    men = people[:n]
    women = people[n:]
    men_prefs = {man: women for man in men}
    women_prefs = {woman: men for woman in women}
    men_free = set(men)
    women_free = set(women)
    engagements = {}

    while men_free:
        man = men_free.pop()
        for woman in men_prefs[man]:
            if woman in women_free:
                engagements[woman] = man
                women_free.remove(woman)
                break
            else:
                curren

In [24]:
# ── Extract source residuals for every code token (one forward pass) ────
from src.goals.subspace_probe import cache_source_residuals

seg_positions = list(range(code_start_tok, len(src_ids)))
print(f"Caching residuals for {len(seg_positions)} positions, layers {cfg_ckpt.layers}...")

all_residuals = cache_source_residuals(mt, source_text, seg_positions, cfg_ckpt.layers)
print("Done.")


Caching residuals for 183 positions, layers [45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]...
Done.


In [25]:
# # ── Verbalize at each token position ────────────────────────────────────
# from src.goals.subspace_probe import SubspaceCache
# from tqdm.auto import tqdm

# MAX_NEW_TOKENS = 32

# patchscope_results = []
# for pos in tqdm(seg_positions, desc="verbalizing"):
#     sc_tmp = SubspaceCache(
#         task=EVAL_TASK,
#         block_kind="",
#         label="",
#         src_residuals=all_residuals[pos],
#         token_pos=pos,
#         is_preceding=False,
#     )
#     gen = verbalize(probe_ckpt, sc_tmp, max_new_tokens=MAX_NEW_TOKENS)
#     patchscope_results.append({"pos": pos, "token": src_tokens[pos], "patched_gen": gen})

# print(f"Collected {len(patchscope_results)} verbalizations.")


In [26]:
from src.goals.subspace_probe import verbalize_batch

MAX_NEW_TOKENS = 32
CHUNK_SIZE     = 16   # reduce if OOM

src_residuals_list = [all_residuals[pos] for pos in seg_positions]

verbalizations = verbalize_batch(
    mt, probe_ckpt,
    before_soft_ids, after_soft_ids,
    src_residuals_list,
    max_new_tokens=MAX_NEW_TOKENS,
    chunk_size=CHUNK_SIZE,
)

patchscope_results = [
    {"pos": pos, "token": src_tokens[pos], "patched_gen": gen}
    for pos, gen in zip(seg_positions, verbalizations)
]
print(f"Collected {len(patchscope_results)} verbalizations.")

Collected 183 verbalizations.


In [27]:
# ── Visualize ────────────────────────────────────────────────────────────
import os
from src.goals.goalscope import show_patchscope

os.makedirs("results/patchscope", exist_ok=True)
save_name = f"subspace_{EVAL_TASK}_{CKPT_NAME.replace('.pt', '')}"
save_path = f"results/patchscope/{save_name}.html"

show_patchscope(
    patchscope_results,
    title=f"Subspace Probe: {EVAL_TASK}  ({CKPT_NAME})",
    save_path=save_path,
)


Saved -> results/patchscope/subspace_stable_marriage_probe_epoch0004.html


In [29]:
from src.functional import generate_with_patch

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant",
    },
    {
        "role": "user",
        "content": "Give me a short rhyme",
    },
    {
        "role": "assistant",
        "content": "He saw a carrot and he had to grab it.\n",
    },
]

rhyme_prompt = mt.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,  # Don't add a new assistant header
    continue_final_message=True,  # Don't close the assistant turn with <|eot_id|>
    tokenize=False,
    chat_template_kwargs={"enable_thinking": False},  # only Qwen3 and Qwen3.5
)

rhyme_prompt_tokenized = prepare_input(rhyme_prompt, tokenizer=mt, return_offsets_mapping=True)
offsets = rhyme_prompt_tokenized.pop("offset_mapping")[0]

rhyme_gen = generate_with_patch(
    mt = mt,
    inputs = rhyme_prompt_tokenized,
    max_new_tokens=32,
    remove_prefix=True,
    do_sample=False
)[0]

rich_print(f"[white]{rhyme_prompt}[/white][bright_yellow]{rhyme_gen}[/bright_yellow]")

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Give me a short rhyme<|im_end|>
<|im_start|>assistant
He saw a carrot and he had to grab it.
In the garden, it was quite a fab hit!

In [30]:
rhyme_full_text = rhyme_prompt + rhyme_gen
rhyme_tokenized = prepare_input(rhyme_full_text, tokenizer=mt)
seg_positions = list(range(rhyme_tokenized["input_ids"].shape[1]))
src_tokens = [mt.tokenizer.decode([t]) for t in rhyme_tokenized["input_ids"][0]]

all_residuals = cache_source_residuals(
    mt, rhyme_full_text, seg_positions, cfg_ckpt.layers
)

src_residuals_list = [all_residuals[pos] for pos in seg_positions]

verbalizations = verbalize_batch(
    mt, probe_ckpt,
    before_soft_ids, after_soft_ids,
    src_residuals_list,
    max_new_tokens=MAX_NEW_TOKENS,
    chunk_size=CHUNK_SIZE,
)

patchscope_results = [
    {"pos": pos, "token": src_tokens[pos], "patched_gen": gen}
    for pos, gen in zip(seg_positions, verbalizations)
]
print(f"Collected {len(patchscope_results)} verbalizations.")

Collected 45 verbalizations.


In [31]:
show_patchscope(
    patchscope_results,
    title=f"Subspace Probe: {EVAL_TASK}  ({CKPT_NAME})",
    # save_path=save_path,
)